# 04 — Text Embeddings with LanceDB (KodeKloud Airlines)

In this notebook we will:

- Load a **policy document** for *KodeKloud Airlines*
- Split it into small chunks
- Convert each chunk into a **vector embedding** (text → numbers)
- Store vectors in **LanceDB**
- Ask **5 questions** (easy → tough) and retrieve the best matching policy chunks

> Goal: build intuition for *why* embeddings + vector search can answer questions from text, even when wording differs.

## 1) Install/import the tools

We’ll use:
- `sentence-transformers` to generate embeddings locally (no API keys)
- `lancedb` to store vectors + run similarity search

In [ ]:
import os
import re
from pathlib import Path

import lancedb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

## 2) Load the KodeKloud Airlines policy document

We’ll treat this policy as our “knowledge base”. In real systems this could be PDFs, wikis, tickets, etc.

In [ ]:
BASE_DIR = Path.cwd()
POLICY_PATH = BASE_DIR / "kodekloud_airlines_policy.md"

policy_text = POLICY_PATH.read_text(encoding="utf-8")
print(f"Loaded policy: {POLICY_PATH.name}")
print(f"Characters: {len(policy_text):,}")

print("\n--- Preview (first 400 chars) ---\n")
print(policy_text[:400])

## 3) Chunk the document into smaller pieces

Vector search works best when we store **small-ish chunks** instead of one huge document.

Here we split by headings (`##` / `###`). Each chunk keeps its section title so we can show context in results.

In [ ]:
def chunk_by_headings(md: str) -> list[dict]:
    # Split on markdown headings, keeping the heading text.
    # This is intentionally simple for teaching.
    parts = re.split(r"(?m)^((?:##|###)\s+.+)$", md)

    chunks: list[dict] = []
    current_title = "(start)"

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part.startswith("## ") or part.startswith("### "):
            current_title = part.lstrip("# ").strip()
            continue

        text = part.strip()
        if len(text) < 60:
            continue

        chunks.append({
            "section": current_title,
            "text": text,
        })

    return chunks

chunks = chunk_by_headings(policy_text)
print(f"Chunks: {len(chunks)}")
print("\nExample chunk:")
print("Section:", chunks[0]["section"])
print("Text (first 250 chars):\n", chunks[0]["text"][:250])

## 4) Turn text into vectors (embeddings)

Embeddings are just lists of numbers where **similar meanings** end up **close together**.

We’ll load a small, fast model and embed:
- One example sentence
- All policy chunks

In [ ]:
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

example = "What is the cabin baggage limit?"
v = model.encode(example)
print("Model:", MODEL_NAME)
print("Vector shape:", v.shape)
print("Vector preview:", np.array2string(v[:10], precision=3))

## 5) Store embeddings in LanceDB

We’ll create a local LanceDB database folder and store rows like:
- `section`: where the chunk came from
- `text`: the chunk content
- `vector`: the embedding for that chunk

Then LanceDB can do similarity search with a single line.

In [ ]:
DB_PATH = str(BASE_DIR / "lancedb_kodekloud_airline")
TABLE_NAME = "policy_chunks"

# Recreate DB folder each run for a clean demo (optional).
# If you want persistence between runs, comment out the next 2 lines.
if Path(DB_PATH).exists():
    import shutil
    shutil.rmtree(DB_PATH)

db = lancedb.connect(DB_PATH)

texts = [c["text"] for c in chunks]
sections = [c["section"] for c in chunks]

vectors = model.encode(texts, normalize_embeddings=True)
rows = [
    {"section": sections[i], "text": texts[i], "vector": vectors[i].tolist()}
    for i in range(len(texts))
]

if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)

tbl = db.create_table(TABLE_NAME, data=rows)
print("Rows in table:", tbl.count_rows())

## 6) Ask questions (vector search)

We’ll embed the question, then ask LanceDB for the **most similar** chunk(s).

Important idea: we’re not doing keyword matching; we’re doing *semantic similarity*.

In [ ]:
def search_policy(question: str, k: int = 3) -> pd.DataFrame:
    qvec = model.encode(question, normalize_embeddings=True).tolist()
    df = (
        tbl.search(qvec)
        .limit(k)
        .to_pandas()
    )
    # LanceDB returns a distance/score column; keep the view clean for teaching.
    cols = [c for c in df.columns if c in {"section", "text", "_distance", "score"}]
    return df[cols]

def pretty_print_results(question: str, k: int = 2, preview_chars: int = 500) -> None:
    print("Question:", question)
    results = search_policy(question, k=k)
    for i, row in results.iterrows():
        section = row.get("section", "")
        text = row.get("text", "")
        dist = row.get("_distance", None)
        print("\n--- Match", i + 1, "---")
        if dist is not None:
            print("Distance:", float(dist))
        print("Section:", section)
        print(text[:preview_chars])

pretty_print_results("What is the cabin baggage weight limit?", k=2)

## 7) Five questions (easy → tough)

Run the next **five cells one-by-one**. Each cell asks a single question so you can observe the results changing.

In [ ]:
# Q1 (easy)
pretty_print_results("What is the cabin baggage weight limit?", k=2, preview_chars=600)

In [ ]:
# Q2
pretty_print_results("If I cancel 30 hours before departure, what refund do I get?", k=2, preview_chars=600)

In [ ]:
# Q3
pretty_print_results("How can I contact KodeKloud Airlines support?", k=2, preview_chars=600)

In [ ]:
# Q4
pretty_print_results("What is the unaccompanied minor service fee?", k=2, preview_chars=600)

In [ ]:
# Q5 (tough)
pretty_print_results("Can I take a dog in the cabin if my pet + carrier is 10 kg?", k=2, preview_chars=600)

## 8) One more question (no shared keywords)

This is where embeddings shine.

The next question avoids the exact wording used in the policy (no “support”, “contact”, or “hours”). Even then, vector search can still retrieve the right chunk because embeddings capture **meaning** (synonyms + related context), not exact keywords.

In [ ]:
# Q6 (no shared keywords)
pretty_print_results(
    "When is your helpdesk staffed so I can reach a human for assistance?",
    k=2,
    preview_chars=600,
)